# Ensemble Optimization: Voting & Stacking

## Objective

Evaluate whether Voting and Stacking ensembles can outperform the tuned **Gradient Boosting** model.

## Results

| Model | Validation AP | Train AP | Test AP |
| --- | ---: | ---: | ---: |
| **Gradient Boosting (Solo)** | **0.095786** | 0.111248 | 0.114806 |
| Stacking (3 Models) | 0.095532 | 0.109498 | 0.112779 |
| Stacking (2 Models) | 0.094438 | 0.115200 | 0.117228 |
| Voting (2 Models, Equal) | 0.093129 | 0.108768 | 0.121577 |
| Voting (2 Models, Weighted) | 0.092585 | 0.111619 | **0.123339** |
| Voting (3 Models, Weighted) | 0.092084 | 0.106342 | 0.114117 |
| Voting (3 Models, Equal) | 0.091174 | 0.101071 | 0.108527 |

## Conclusion

- None of the Voting or Stacking models outperformed the tuned **Gradient Boosting** model on Validation AP.
- Although some Voting models achieved higher Test AP, their Validation AP was lower, indicating inconsistent generalization.
- The performance gain from ensemble methods is negligible and does not justify the added complexity.

## Final Recommendation

**Gradient Boosting (Solo)** remains the final model. It provides the best overall performance and generalization, making Voting and Stacking unnecessary for this project.

In [16]:
import warnings
warnings.filterwarnings("ignore")

In [17]:
from pathlib import Path
import sys

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root))

In [35]:
READ_CSV="../../data/interim/data_travel_insurance_interim.csv"
SAVE_RESULT = "results/voting_stacking_optimization.csv"
RANDOM_STATE=42


TARGET_TRANSFORM_COLS = ["Destination"]
LOG_TRANSFORM_COLS= ["Duration", "Net Sales"]

In [19]:
from src.utils import benchmark_models

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder, TargetEncoder
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import PowerTransformer
from sklearn.feature_selection import SelectFdr, SelectFromModel, f_classif
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

from feature_engine.outliers import Winsorizer

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

In [20]:
df = pd.read_csv(READ_CSV)
df.head()

,Agency,Agency Type,Distribution Channel,Product Name,Duration,Destination,Net Sales,Commission,Age,Claim,Is Refund,Suspected Fraud,Commission Rate
0,C2B,Airlines,Online,Annual Silver Plan,365,SINGAPORE,216.0,54.0,57,0,No,No,0.25
1,EPX,Travel Agency,Online,Cancellation Plan,4,MALAYSIA,10.0,0.0,33,0,No,No,0.00
2,JZI,Airlines,Online,Basic Plan,19,INDIA,22.0,7.7,26,0,No,No,0.35
3,EPX,Travel Agency,Online,2 way Comprehensive Plan,20,UNITED STATES,112.0,0.0,59,0,No,No,0.00
4,C2B,Airlines,Online,Bronze Plan,8,SINGAPORE,16.0,4.0,28,0,No,No,0.25


In [21]:
x = df.drop(columns=["Claim"])
y = df["Claim"]


In [22]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [23]:
NUMERIC_COLS = [features for features in x_train.columns if x_train[features].dtypes != "O"]
CATEGORICAL_COLS = [features for features in x_train.columns if x_train[features].dtypes == "O"]


In [24]:
numeric_pipeline = Pipeline([
    ("winsorizer_iqr", Winsorizer(capping_method="iqr", fold=1.5)),
    ("RobustScaler", RobustScaler()),
])

numeric_log_pipeline = Pipeline([
    ("power", PowerTransformer(method="yeo-johnson")),
    ("RobustScaler", RobustScaler()),
])

categorical_ohe_pipeline = Pipeline([
     ("OneHotEncoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"))
])

categorical_target_pipeline = Pipeline([
     ("TargetEncoder", TargetEncoder())
])


In [25]:
preprocessor = ColumnTransformer(
    [
        ("numeric_pipeline", numeric_pipeline, [c for c in NUMERIC_COLS if c not in LOG_TRANSFORM_COLS]),
        ("numeric_log_pipeline", numeric_log_pipeline, LOG_TRANSFORM_COLS),
        ("categorical_ohe_pipeline", categorical_ohe_pipeline, [c for c in CATEGORICAL_COLS if c not in TARGET_TRANSFORM_COLS]),
        ("categorical_target_pipeline", categorical_target_pipeline, TARGET_TRANSFORM_COLS),
    ],
    remainder="drop"
)

In [26]:
gb_pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("feature_selection", SelectFdr(score_func=f_classif, alpha=0.005)),
    ("resampler", RandomOverSampler(random_state=RANDOM_STATE)),
    ("classifier", GradientBoostingClassifier(random_state=RANDOM_STATE))
])
gb_best_params = {
    'classifier__learning_rate': 0.020585312110064476,
    'classifier__max_depth': 4,
    'classifier__max_features': 'log2',
    'classifier__min_samples_leaf': 26,
    'classifier__min_samples_split': 42,
    'classifier__n_estimators': 177,
    'classifier__subsample': 0.8227412591061039
}
gb_pipeline.set_params(**gb_best_params)

,steps,"[('preprocessor', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_pipeline', ...), ('numeric_log_pipeline', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [27]:
rf_pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("feature_selection", SelectFromModel(
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        threshold="1.5*mean"
    )),
    ("resampler", "passthrough"),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE))
])
rf_best_params = {
    'classifier__bootstrap': True,
    'classifier__ccp_alpha': 3.533152609858703e-05,
    'classifier__max_depth': 4,
    'classifier__max_features': 'sqrt',
    'classifier__min_samples_leaf': 41,
    'classifier__min_samples_split': 78,
    'classifier__n_estimators': 269
}
rf_pipeline.set_params(**rf_best_params)

,steps,"[('preprocessor', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_pipeline', ...), ('numeric_log_pipeline', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [28]:
lr_pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("feature_selection", SelectFdr(score_func=f_classif, alpha=0.05)),
    ("resampler", "passthrough"),
    ("classifier", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        random_state=RANDOM_STATE
    ))
])
lr_best_params = {
    'classifier__C': 0.4524488502720415,
    'classifier__class_weight': None,
    'classifier__fit_intercept': True
}
lr_pipeline.set_params(**lr_best_params)

,steps,"[('preprocessor', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric_pipeline', ...), ('numeric_log_pipeline', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
ensemble_experiments = {
    "Voting_3Models_Equal": VotingClassifier(
        estimators=[
            ("GradientBoost", gb_pipeline),
            ("RandomForest", rf_pipeline),
            ("LogisticRegressionLasso", lr_pipeline)
        ],
        voting="soft",
        n_jobs=-1
    ),
    "Voting_3Models_Weighted": VotingClassifier(
        estimators=[
            ("GradientBoost", gb_pipeline),
            ("RandomForest", rf_pipeline),
            ("LogisticRegressionLasso", lr_pipeline)
        ],
        voting="soft",
        weights=[5, 4, 2],
        n_jobs=-1
    ),
    "Voting_2Models_Equal": VotingClassifier(
        estimators=[
            ("GradientBoost", gb_pipeline),
            ("RandomForest", rf_pipeline)
        ],
        voting="soft",
        n_jobs=-1
    ),
    "Voting_2Models_Weighted": VotingClassifier(
        estimators=[
            ("GradientBoost", gb_pipeline),
            ("RandomForest", rf_pipeline)
        ],
        voting="soft",
        weights=[3, 2],
        n_jobs=-1
    ),
    
    "Stacking_3Models": StackingClassifier(
        estimators=[
            ("GradientBoost", gb_pipeline),
            ("RandomForest", rf_pipeline),
            ("LogisticRegressionLasso", lr_pipeline)
        ],
        final_estimator=LogisticRegression(random_state=RANDOM_STATE),
        cv=5,
        n_jobs=-1
    ),
    
    "Stacking_2Models": StackingClassifier(
        estimators=[
            ("GradientBoost", gb_pipeline),
            ("RandomForest", rf_pipeline)
        ],
        final_estimator=LogisticRegression(random_state=RANDOM_STATE),
        cv=5,
        n_jobs=-1
    )
}

In [30]:
ensemble_results = []

for model_name, clf in ensemble_experiments.items():
    cv_scores = cross_validate(
        clf, 
        x_train, 
        y_train, 
        cv=5, 
        scoring="average_precision", 
        n_jobs=-1
    )
    val_ap = np.mean(cv_scores["test_score"])
    
    clf.fit(x_train, y_train)
    
    train_prob = clf.predict_proba(x_train)[:, 1]
    test_prob = clf.predict_proba(x_test)[:, 1]
    
    train_ap = average_precision_score(y_train, train_prob)
    test_ap = average_precision_score(y_test, test_prob)
    
    ensemble_results.append({
        "Model": model_name,
        "ap_validate_score": val_ap,
        "ap_train_score": train_ap,
        "ap_test_score": test_ap
    })
    
df_ensemble_results = pd.DataFrame(ensemble_results)
df_ensemble_results = df_ensemble_results.sort_values(by="ap_validate_score", ascending=False).reset_index(drop=True)

In [31]:
df_ensemble_results

,Model,ap_validate_score,ap_train_score,ap_test_score
0,Stacking_3Models,0.095532,0.109498,0.112779
1,Stacking_2Models,0.094438,0.115200,0.117228
2,Voting_2Models_Equal,0.093129,0.108768,0.121577
3,Voting_2Models_Weighted,0.092585,0.111619,0.123339
4,Voting_3Models_Weighted,0.092084,0.106342,0.114117
5,Voting_3Models_Equal,0.091174,0.101071,0.108527


In [36]:
df_ensemble_results.to_csv(SAVE_RESULT, index=False)